# Phase 4: Dynamic Tone and Detail Modulation with Few-Shot Prompting for MedChat

This notebook implements a dynamic prompt generation framework with real-time adaptation mechanisms for tone adjustment and detail modulation using few-shot prompting.

## Features:
- Context detection (urgency, anxiety, knowledge level, cultural sensitivity)
- Few-shot example bank with multiple tones and detail levels
- Dynamic prompt generation
- Integration with existing Mistral-7B model
- Real-time adaptation based on patient context

In [1]:
# Import necessary libraries
import torch
import json
import re
import numpy as np
from typing import Dict, List, Tuple, Optional
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    pipeline
)
from peft import PeftModel
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Configuration and setup 
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"  
ADAPTER_PATH = "./model_output"  # Path to your trained adapter
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

Device: cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
GPU Memory: 7.6 GB
BF16 supported: True


In [3]:
# Load the model and tokenizer - Simplified approach matching your training setup
def load_model_and_tokenizer():
    """Load the Mistral-7B model with trained adapter"""
    
    print(f"Loading model: {MODEL_NAME}")
    
    # Configure quantization - same as your training notebook
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    )
    
    # Load tokenizer - same setup as training
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    # Load base model - direct approach like your training notebook
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        use_cache=False
    )
    
    # Load adapter if it exists
    try:
        model = PeftModel.from_pretrained(model, ADAPTER_PATH)
        print(" Loaded model with trained adapter")
    except:
        print("Using base model only (adapter not found - this is normal if you haven't trained yet)")
    
    return model, tokenizer

# Load the model
print("Loading model and tokenizer...")
model, tokenizer = load_model_and_tokenizer()
print("Model loaded successfully!")

Loading model and tokenizer...
Loading model: mistralai/Mistral-7B-Instruct-v0.3


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

 Loaded model with trained adapter
Model loaded successfully!


## Context Detection System

This system analyzes patient input to determine urgency, anxiety levels, medical knowledge, and cultural sensitivity requirements.

In [4]:
class PatientContext:
    """Class to hold patient context information"""
    
    def __init__(self, urgency_score: float, anxiety_level: float, knowledge_level: str, 
                 formality_preference: str, detected_keywords: List[str], 
                 tone_type: str = '', detail_level: str = ''):
        self.urgency_score = urgency_score  # 0-1 scale
        self.anxiety_level = anxiety_level  # 0-1 scale
        self.knowledge_level = knowledge_level  # 'basic', 'intermediate', 'advanced'
        self.formality_preference = formality_preference  # 'formal', 'casual'
        self.detected_keywords = detected_keywords
        self.tone_type = tone_type  # Will be determined later
        self.detail_level = detail_level  # Will be determined later

def detect_context(patient_input: str, conversation_history: List[str] = None) -> PatientContext:
    """Detect patient context from input and conversation history"""
    
    if conversation_history is None:
        conversation_history = []
    
    # Combine current input with recent history for better context
    full_text = patient_input.lower()
    if conversation_history:
        # Use last 3 messages for context
        recent_history = " ".join(conversation_history[-3:])
        full_text = f"{recent_history.lower()} {full_text}"
    
    # Urgency detection keywords and scoring
    urgency_keywords = {
        'critical': ['emergency', 'urgent', 'critical', 'severe', 'extreme', 'intense'],
        'high': ['blood', 'bleeding', 'chest pain', 'difficulty breathing', 'can\'t breathe', 
                'shortness of breath', 'severe pain', 'unconscious', 'seizure'],
        'medium': ['pain', 'fever', 'nausea', 'vomiting', 'dizzy', 'headache'],
        'low': ['mild', 'slight', 'minor', 'occasional']
    }
    
    urgency_score = 0.0
    detected_keywords = []
    
    for level, keywords in urgency_keywords.items():
        for keyword in keywords:
            if keyword in full_text:
                detected_keywords.append(keyword)
                if level == 'critical':
                    urgency_score = max(urgency_score, 1.0)
                elif level == 'high':
                    urgency_score = max(urgency_score, 0.8)
                elif level == 'medium':
                    urgency_score = max(urgency_score, 0.5)
                elif level == 'low':
                    urgency_score = max(urgency_score, 0.2)
    
    # Anxiety detection
    anxiety_markers = [
        'worried', 'scared', 'anxious', 'concerned', 'frightened', 'nervous',
        'terrified', 'panic', 'afraid', 'fear', 'stress', 'overwhelmed'
    ]
    
    anxiety_level = 0.0
    for marker in anxiety_markers:
        if marker in full_text:
            detected_keywords.append(marker)
            anxiety_level = min(anxiety_level + 0.3, 1.0)
    
    # Medical knowledge level assessment
    medical_terms = [
        'diagnosis', 'prognosis', 'etiology', 'pathophysiology', 'differential',
        'contraindication', 'pharmacokinetics', 'therapeutic', 'clinical',
        'syndrome', 'pathology', 'epidemiology'
    ]
    
    basic_terms = [
        'what is', 'explain', 'simple terms', 'don\'t understand',
        'layman', 'plain English', 'basic'
    ]
    
    medical_term_count = sum(1 for term in medical_terms if term in full_text)
    basic_term_count = sum(1 for term in basic_terms if term in full_text)
    
    if medical_term_count >= 2:
        knowledge_level = 'advanced'
    elif medical_term_count >= 1 and basic_term_count == 0:
        knowledge_level = 'intermediate'
    else:
        knowledge_level = 'basic'
    
    # Cultural sensitivity / formality detection
    formal_indicators = ['please', 'thank you', 'sir', 'madam', 'doctor']
    casual_indicators = ['hey', 'hi', 'yeah', 'ok', 'thanks']
    
    formal_count = sum(1 for indicator in formal_indicators if indicator in full_text)
    casual_count = sum(1 for indicator in casual_indicators if indicator in full_text)
    
    formality_preference = 'formal' if formal_count > casual_count else 'casual'
    
    return PatientContext(
        urgency_score=urgency_score,
        anxiety_level=anxiety_level,
        knowledge_level=knowledge_level,
        formality_preference=formality_preference,
        detected_keywords=detected_keywords,
        tone_type='',  # Will be set by determine_tone
        detail_level=''  # Will be set by determine_detail_level
    )

# Test the context detection
test_inputs = [
    "I'm having severe chest pain and can't breathe properly",
    "I'm really worried about this mild headache I've been having",
    "Can you explain the pathophysiology of myocardial infarction?",
    "What is diabetes? Please explain in simple terms"
]

print("Testing context detection:")
for i, test_input in enumerate(test_inputs, 1):
    context = detect_context(test_input)
    print(f"\n{i}. Input: {test_input}")
    print(f"   Urgency: {context.urgency_score:.2f}")
    print(f"   Anxiety: {context.anxiety_level:.2f}")
    print(f"   Knowledge: {context.knowledge_level}")
    print(f"   Formality: {context.formality_preference}")
    print(f"   Keywords: {context.detected_keywords}")

Testing context detection:

1. Input: I'm having severe chest pain and can't breathe properly
   Urgency: 1.00
   Anxiety: 0.00
   Knowledge: basic
   Formality: casual
   Keywords: ['severe', 'chest pain', "can't breathe", 'pain']

2. Input: I'm really worried about this mild headache I've been having
   Urgency: 0.50
   Anxiety: 0.30
   Knowledge: basic
   Formality: casual
   Keywords: ['headache', 'mild', 'worried']

3. Input: Can you explain the pathophysiology of myocardial infarction?
   Urgency: 0.00
   Anxiety: 0.00
   Knowledge: basic
   Formality: casual
   Keywords: []

4. Input: What is diabetes? Please explain in simple terms
   Urgency: 0.00
   Anxiety: 0.00
   Knowledge: basic
   Formality: formal
   Keywords: []


## Few-Shot Example Bank

Comprehensive database of medical conversation examples across different tones and detail levels.

In [5]:
# Few-shot example bank
EXAMPLE_BANK = {
    "urgent_professional": {
        "high_detail": [
            {
                "input": "I'm having severe chest pain and shortness of breath",
                "response": "This requires immediate medical attention. Chest pain with shortness of breath can indicate several serious conditions including myocardial infarction, pulmonary embolism, or aortic dissection. Please call emergency services immediately (911) or go to the nearest emergency room. Do not drive yourself. While waiting for help, sit upright, loosen tight clothing, and if you have prescribed nitroglycerin, take it as directed. Time is critical in cardiac events."
            },
            {
                "input": "There's blood in my vomit",
                "response": "Hematemesis (blood in vomit) is a medical emergency requiring immediate evaluation. This can indicate upper gastrointestinal bleeding from conditions such as peptic ulcers, esophageal varices, or Mallory-Weiss tears. Please seek emergency medical care immediately. Do not eat or drink anything. Note the color and amount of blood, and bring this information to medical personnel. Call 911 if you feel dizzy, weak, or have rapid heartbeat."
            }
        ],
        "medium_detail": [
            {
                "input": "I'm having severe chest pain and shortness of breath",
                "response": "This is a medical emergency. Chest pain with breathing difficulties can be signs of a heart attack or other serious conditions. Call 911 immediately or go to the emergency room right away. Don't drive yourself. Sit up straight and loosen any tight clothing while you wait for help. If you have heart medication like nitroglycerin, take it now."
            },
            {
                "input": "There's blood in my vomit",
                "response": "Blood in vomit is serious and needs immediate medical attention. This could indicate bleeding in your stomach or esophagus. Please go to the emergency room right away. Don't eat or drink anything. Try to remember how much blood you saw and what color it was - this information will help doctors. Call 911 if you feel faint or your heart is racing."
            }
        ]
    },
    "reassuring_empathetic": {
        "high_detail": [
            {
                "input": "I'm really worried about this headache",
                "response": "I understand your concern about the headache, and it's completely natural to feel worried when experiencing persistent symptoms. Most headaches are benign and caused by tension, dehydration, stress, or minor sinus issues. However, let's discuss the characteristics to better assess your situation. Can you describe the pain quality (throbbing, sharp, dull), location, duration, and any associated symptoms like visual changes, nausea, or fever? Red flags that would require immediate attention include sudden severe onset, fever with neck stiffness, or neurological symptoms."
            },
            {
                "input": "I'm scared about these test results",
                "response": "I completely understand how frightening it can be to receive unexpected test results, and your feelings are entirely valid. Anxiety in these situations is a normal human response. Let's work through this together step by step. First, it's important to remember that abnormal results don't always indicate serious pathology - they often require additional testing for clarification or may represent minor variations. Can you share what specific results are concerning you? Understanding the context will help us discuss the clinical significance and next steps."
            }
        ],
        "medium_detail": [
            {
                "input": "I'm really worried about this headache",
                "response": "I understand you're worried, and that's completely normal when dealing with persistent symptoms. Most headaches are not serious and are often caused by stress, dehydration, or tension. To help you better, can you tell me more about your headache - when it started, what it feels like, and if you have any other symptoms? There are some warning signs to watch for, but the majority of headaches resolve with rest and basic care."
            },
            {
                "input": "I'm scared about these test results",
                "response": "It's completely understandable to feel scared about test results - that's a very normal reaction. Remember that abnormal results don't always mean something serious is wrong. Sometimes they just need more investigation or might be minor variations. Can you tell me which results are worrying you? Once I understand what you're seeing, I can help explain what they might mean and what the typical next steps would be."
            }
        ],
        "low_detail": [
            {
                "input": "I'm really worried about this headache",
                "response": "I can hear that you're worried, and that's completely understandable. Most headaches aren't anything serious - they're usually from things like stress, not drinking enough water, or tension. Can you tell me a bit more about when it started and what it feels like? I'm here to help you figure this out, and we'll take it one step at a time."
            },
            {
                "input": "I'm scared about these test results",
                "response": "I can really understand why you'd feel scared - getting test results can be really overwhelming. The good news is that unusual results don't always mean something bad is happening. Sometimes they just need a closer look. What results are you most worried about? I'm here to help you understand what they might mean."
            }
        ]
    },
    "standard_professional": {
        "high_detail": [
            {
                "input": "What causes diabetes?",
                "response": "Diabetes mellitus results from defects in insulin production, insulin action, or both, leading to chronic hyperglycemia. Type 1 diabetes is an autoimmune condition where pancreatic beta cells are destroyed, typically presenting in childhood with absolute insulin deficiency. Type 2 diabetes involves insulin resistance in peripheral tissues combined with progressive beta cell dysfunction, often associated with metabolic syndrome, obesity, and genetic predisposition. Gestational diabetes occurs during pregnancy due to insulin resistance caused by placental hormones. The pathophysiology involves disrupted glucose homeostasis, leading to complications affecting the cardiovascular, renal, and nervous systems through advanced glycation end products and oxidative stress."
            },
            {
                "input": "How does blood pressure medication work?",
                "response": "Antihypertensive medications work through various mechanisms targeting the renin-angiotensin-aldosterone system, sympathetic nervous system, and vascular smooth muscle. ACE inhibitors block angiotensin-converting enzyme, reducing angiotensin II formation and decreasing vasoconstriction and aldosterone secretion. ARBs directly antagonize angiotensin II receptors. Beta-blockers reduce cardiac output and renin release by blocking adrenergic receptors. Calcium channel blockers prevent calcium influx into vascular smooth muscle cells, causing vasodilation. Diuretics reduce blood volume through natriuresis and have direct vasodilatory effects through mechanisms involving prostaglandins and nitric oxide."
            }
        ],
        "medium_detail": [
            {
                "input": "What causes diabetes?",
                "response": "Diabetes occurs when your body can't properly use or make insulin, a hormone that helps control blood sugar. There are different types: Type 1 happens when the immune system attacks insulin-producing cells in the pancreas, usually starting in childhood. Type 2 develops when your body becomes resistant to insulin or doesn't make enough, often linked to genetics, weight, and lifestyle. Gestational diabetes occurs during pregnancy. All types result in high blood sugar, which over time can damage blood vessels, nerves, and organs if not well controlled."
            },
            {
                "input": "How does blood pressure medication work?",
                "response": "Blood pressure medications work in different ways to lower the pressure in your arteries. Some (like ACE inhibitors) relax blood vessels by blocking hormones that tighten them. Others (beta-blockers) slow your heart rate and reduce the force of heartbeats. Calcium channel blockers prevent vessels from tightening by blocking calcium. Diuretics help your kidneys remove excess water and salt, reducing blood volume. Your doctor chooses based on your specific condition and how your body responds."
            }
        ],
        "low_detail": [
            {
                "input": "What causes diabetes?",
                "response": "Diabetes happens when your body has trouble controlling blood sugar. Think of insulin as a key that lets sugar into your cells for energy. In Type 1 diabetes, your body stops making this key. In Type 2, either you don't make enough keys or the locks on your cells don't work properly. This causes sugar to build up in your blood instead of getting into cells where it's needed. It can be caused by genetics, weight, or lifestyle factors."
            },
            {
                "input": "How does blood pressure medication work?",
                "response": "Blood pressure medications help lower the pressure of blood flowing through your arteries, kind of like turning down the water pressure in a garden hose. Different medicines work in different ways - some help your blood vessels relax and widen, others slow down your heart rate, and some help your body get rid of extra water. Your doctor picks the best type based on what will work best for you."
            }
        ]
    }
}

print("Few-shot example bank loaded successfully!")
print(f"Total examples: {sum(len(examples) for tone in EXAMPLE_BANK.values() for examples in tone.values())}")

Few-shot example bank loaded successfully!
Total examples: 16


## Tone and Detail Level Determination

Functions to determine appropriate tone and detail level based on patient context.

In [6]:
def determine_tone(urgency_score: float, anxiety_level: float) -> str:
    """Determine appropriate tone based on urgency and anxiety levels"""
    
    if urgency_score >= 0.7:
        return "urgent_professional"
    elif anxiety_level >= 0.5:
        return "reassuring_empathetic"
    else:
        return "standard_professional"

def determine_detail_level(knowledge_level: str) -> str:
    """Determine appropriate detail level based on knowledge assessment"""
    
    detail_mapping = {
        'basic': 'low_detail',
        'intermediate': 'medium_detail',
        'advanced': 'high_detail'
    }
    
    return detail_mapping.get(knowledge_level, 'medium_detail')

def get_relevant_examples(tone_type: str, detail_level: str, query_topic: str = None) -> List[Dict]:
    """Get relevant few-shot examples based on tone and detail level"""
    
    if tone_type not in EXAMPLE_BANK:
        tone_type = "standard_professional"
    
    if detail_level not in EXAMPLE_BANK[tone_type]:
        detail_level = "medium_detail"
    
    examples = EXAMPLE_BANK[tone_type][detail_level]
    
    # If query_topic is provided, try to find most relevant examples
    if query_topic:
        query_topic = query_topic.lower()
        scored_examples = []
        
        for example in examples:
            # Simple relevance scoring based on keyword overlap
            input_words = set(example['input'].lower().split())
            query_words = set(query_topic.split())
            overlap = len(input_words.intersection(query_words))
            scored_examples.append((overlap, example))
        
        # Sort by relevance and return top examples
        scored_examples.sort(key=lambda x: x[0], reverse=True)
        return [example for _, example in scored_examples[:2]]  # Return top 2
    
    return examples[:2]  # Return first 2 examples if no topic specified

# Test tone and detail determination
test_contexts = [
    PatientContext(0.9, 0.3, 'basic', 'formal', ['severe', 'chest pain'], '', ''),
    PatientContext(0.2, 0.8, 'intermediate', 'casual', ['worried', 'scared'], '', ''),
    PatientContext(0.1, 0.1, 'advanced', 'formal', [], '', ''),
    PatientContext(0.3, 0.4, 'basic', 'casual', ['headache'], '', '')
]

print("Testing tone and detail determination:")
for i, context in enumerate(test_contexts, 1):
    tone = determine_tone(context.urgency_score, context.anxiety_level)
    detail = determine_detail_level(context.knowledge_level)
    examples = get_relevant_examples(tone, detail)
    
    print(f"\n{i}. Context: Urgency={context.urgency_score}, Anxiety={context.anxiety_level}, Knowledge={context.knowledge_level}")
    print(f"   Tone: {tone}")
    print(f"   Detail: {detail}")
    print(f"   Examples found: {len(examples)}")

Testing tone and detail determination:

1. Context: Urgency=0.9, Anxiety=0.3, Knowledge=basic
   Tone: urgent_professional
   Detail: low_detail
   Examples found: 2

2. Context: Urgency=0.2, Anxiety=0.8, Knowledge=intermediate
   Tone: reassuring_empathetic
   Detail: medium_detail
   Examples found: 2

3. Context: Urgency=0.1, Anxiety=0.1, Knowledge=advanced
   Tone: standard_professional
   Detail: high_detail
   Examples found: 2

4. Context: Urgency=0.3, Anxiety=0.4, Knowledge=basic
   Tone: standard_professional
   Detail: low_detail
   Examples found: 2


## Few-Shot Prompt Generation System

Creates dynamic prompts with relevant examples based on patient context.

In [7]:
def create_few_shot_prompt(patient_context: PatientContext, user_query: str, tone_type: str, detail_level: str) -> str:
    """Create a few-shot prompt with relevant examples"""
    
    # Get relevant examples
    examples = get_relevant_examples(tone_type, detail_level, user_query)
    
    # Create system prompt based on context
    system_prompt = f"""You are a knowledgeable medical assistant providing helpful, accurate information to patients. 

Context Analysis:
- Urgency Level: {patient_context.urgency_score:.2f}/1.0
- Anxiety Level: {patient_context.anxiety_level:.2f}/1.0
- Medical Knowledge: {patient_context.knowledge_level}
- Communication Style: {patient_context.formality_preference}
- Required Tone: {tone_type.replace('_', ' ').title()}
- Detail Level: {detail_level.replace('_', ' ').title()}

Instructions:
"""
    
    # Add specific instructions based on tone
    if tone_type == "urgent_professional":
        system_prompt += """- This is a URGENT medical situation. Prioritize immediate action and safety.
- Be direct, clear, and authoritative.
- Include specific emergency instructions.
- Emphasize the need for immediate medical attention.
"""
    elif tone_type == "reassuring_empathetic":
        system_prompt += """- The patient is showing signs of anxiety or worry.
- Be compassionate, understanding, and reassuring.
- Acknowledge their feelings and concerns.
- Provide comfort while maintaining medical accuracy.
"""
    else:
        system_prompt += """- Maintain a professional, informative tone.
- Be clear and helpful.
- Provide balanced, evidence-based information.
"""
    
    # Add detail level instructions
    if detail_level == "high_detail":
        system_prompt += "- Provide detailed medical explanations with technical terms.\n- Include pathophysiology and clinical details when appropriate.\n"
    elif detail_level == "low_detail":
        system_prompt += "- Use simple, easy-to-understand language.\n- Avoid medical jargon and complex explanations.\n- Use analogies when helpful.\n"
    else:
        system_prompt += "- Provide moderate detail with some medical terms explained.\n- Balance technical accuracy with accessibility.\n"
    
    # Add examples
    system_prompt += "\nHere are examples of appropriate responses:\n\n"
    
    for i, example in enumerate(examples, 1):
        system_prompt += f"Example {i}:\n"
        system_prompt += f"Patient: {example['input']}\n"
        system_prompt += f"Response: {example['response']}\n\n"
    
    # Add the current query
    system_prompt += f"Now respond to this patient query in the same style:\n\nPatient: {user_query}\nResponse:"
    
    return system_prompt

# Test prompt generation
test_query = "I'm worried about this persistent cough I've had for two weeks"
test_context = detect_context(test_query)
test_context.tone_type = determine_tone(test_context.urgency_score, test_context.anxiety_level)
test_context.detail_level = determine_detail_level(test_context.knowledge_level)

test_prompt = create_few_shot_prompt(test_context, test_query, test_context.tone_type, test_context.detail_level)

print("Sample Few-Shot Prompt:")
print("=" * 80)
print(test_prompt[:1000] + "..." if len(test_prompt) > 1000 else test_prompt)
print("=" * 80)

Sample Few-Shot Prompt:
You are a knowledgeable medical assistant providing helpful, accurate information to patients. 

Context Analysis:
- Urgency Level: 0.00/1.0
- Anxiety Level: 0.30/1.0
- Medical Knowledge: basic
- Communication Style: casual
- Required Tone: Standard Professional
- Detail Level: Low Detail

Instructions:
- Maintain a professional, informative tone.
- Be clear and helpful.
- Provide balanced, evidence-based information.
- Use simple, easy-to-understand language.
- Avoid medical jargon and complex explanations.
- Use analogies when helpful.

Here are examples of appropriate responses:

Example 1:
Patient: What causes diabetes?
Response: Diabetes happens when your body has trouble controlling blood sugar. Think of insulin as a key that lets sugar into your cells for energy. In Type 1 diabetes, your body stops making this key. In Type 2, either you don't make enough keys or the locks on your cells don't work properly. This causes sugar to build up in your blood inste

## Adaptive Response Generation Pipeline

Main function that integrates all components to generate contextually appropriate responses.

In [8]:
def generate_adaptive_response(
    user_input: str, 
    conversation_history: List[str] = None,
    model = None,
    tokenizer = None,
    max_length: int = 512
) -> Tuple[str, PatientContext]:
    """Generate adaptive response based on patient context"""
    
    # Use global model and tokenizer if not provided
    if model is None:
        model = globals()['model']
    if tokenizer is None:
        tokenizer = globals()['tokenizer']
    
    # Step 1: Detect patient context
    patient_context = detect_context(user_input, conversation_history)
    
    # Step 2: Determine tone and detail level
    tone_type = determine_tone(patient_context.urgency_score, patient_context.anxiety_level)
    detail_level = determine_detail_level(patient_context.knowledge_level)
    
    # Update context with determined values
    patient_context.tone_type = tone_type
    patient_context.detail_level = detail_level
    
    # Step 3: Create few-shot prompt
    prompt = create_few_shot_prompt(patient_context, user_input, tone_type, detail_level)
    
    # Step 4: Generate response using the model
    try:
        # Tokenize the prompt
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=2048,
            padding=True
        ).to(model.device)
        
        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_length,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1
            )
        
        # Decode the response
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract only the new response (after the prompt)
        response = full_response[len(prompt):].strip()
        
        # Clean up the response
        if response.startswith("Response:"):
            response = response[9:].strip()
        
        # Ensure response is not empty
        if not response:
            response = "I understand your concern. Could you provide more details so I can better assist you?"
        
        return response, patient_context
        
    except Exception as e:
        print(f"Error generating response: {e}")
        # Fallback response based on context
        if patient_context.urgency_score >= 0.7:
            fallback = "This appears to be a serious medical concern. Please seek immediate medical attention or call emergency services."
        elif patient_context.anxiety_level >= 0.5:
            fallback = "I understand you're concerned about this. While I can't provide a full response right now, please consider consulting with a healthcare provider for proper evaluation."
        else:
            fallback = "I'm here to help, but I'm having trouble generating a response right now. Please rephrase your question or consult with a healthcare provider."
        
        return fallback, patient_context

print("Adaptive response generation pipeline ready!")

Adaptive response generation pipeline ready!


## Testing and Demonstration

Test the complete system with various scenarios to demonstrate dynamic adaptation.

In [9]:
# # Test scenarios covering different contexts
# test_scenarios = [
#     {
#         "name": "Emergency Scenario",
#         "input": "I'm having severe chest pain and can't breathe properly",
#         "expected_tone": "urgent_professional",
#         "expected_detail": "low_detail"
#     },
#     {
#         "name": "Anxious Patient",
#         "input": "I'm really worried and scared about this lump I found",
#         "expected_tone": "reassuring_empathetic",
#         "expected_detail": "low_detail"
#     },
#     {
#         "name": "Medical Professional",
#         "input": "Can you explain the pathophysiology of diabetic nephropathy?",
#         "expected_tone": "standard_professional",
#         "expected_detail": "high_detail"
#     },
#     {
#         "name": "Basic Question",
#         "input": "What is high blood pressure? Please explain in simple terms",
#         "expected_tone": "standard_professional",
#         "expected_detail": "low_detail"
#     },
#     {
#         "name": "Routine Consultation",
#         "input": "I've been having mild headaches lately. What could cause this?",
#         "expected_tone": "standard_professional",
#         "expected_detail": "medium_detail"
#     }
# ]

# def run_test_scenarios():
#     """Run all test scenarios and display results"""
    
#     print(" TESTING DYNAMIC TONE AND DETAIL MODULATION SYSTEM")
#     print("=" * 80)
    
#     for i, scenario in enumerate(test_scenarios, 1):
#         print(f"\n Test {i}: {scenario['name']}")
#         print("-" * 60)
#         print(f" Patient Input: {scenario['input']}")
        
#         # Generate response
#         response, context = generate_adaptive_response(scenario['input'])
        
#         print(f"\n Context Analysis:")
#         print(f"   Urgency Score: {context.urgency_score:.2f}")
#         print(f"   Anxiety Level: {context.anxiety_level:.2f}")
#         print(f"   Knowledge Level: {context.knowledge_level}")
#         print(f"   Formality: {context.formality_preference}")
#         print(f"   Detected Keywords: {context.detected_keywords}")
        
#         print(f"\n Adaptation Results:")
#         print(f"   Tone Type: {context.tone_type}")
#         print(f"   Detail Level: {context.detail_level}")
        
#         # Check if adaptation matches expectations
#         tone_match = context.tone_type == scenario['expected_tone']
#         detail_match = context.detail_level == scenario['expected_detail']
        
#         print(f"   Expected Tone: {scenario['expected_tone']} {'match' if tone_match else 'not match'}")
#         print(f"   Expected Detail: {scenario['expected_detail']} {'match' if detail_match else 'not match'}")

#         print(f"\n Generated Response:")
#         print(f"   {response[:200]}{'...' if len(response) > 200 else ''}")
        
#         print("\n" + "=" * 80)

# # Run the tests
# run_test_scenarios()

## Interactive Demo

Interactive interface to test the system with custom inputs.

In [10]:
def interactive_demo():
    """Interactive demo for testing the adaptive response system"""
    
    print("🩺 MedChat Dynamic Prompting System - Interactive Demo")
    print("=" * 60)
    print("Enter patient queries to see adaptive responses in real-time.")
    print("Type 'quit' to exit the demo.")
    print("Type 'examples' to see sample queries.")
    print("=" * 60)
    
    conversation_history = []
    
    while True:
        user_input = input("\n Patient: ").strip()
        
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Thank you for using MedChat! Goodbye! ")
            break
            
        if user_input.lower() == 'examples':
            print("\n Sample queries to try:")
            sample_queries = [
                "I'm having severe chest pain and shortness of breath",
                "I'm really worried about this headache I've been having",
                "Can you explain the pathophysiology of hypertension?",
                "What is diabetes? Please explain in simple terms",
                "I'm anxious about my upcoming surgery",
                "What are the side effects of blood pressure medication?"
            ]
            for i, query in enumerate(sample_queries, 1):
                print(f"   {i}. {query}")
            continue
        
        if not user_input:
            print("Please enter a query or type 'quit' to exit.")
            continue
        
        try:
            # Generate adaptive response
            response, context = generate_adaptive_response(user_input, conversation_history)
            
            # Display context analysis
            print(f"\n Context: Urgency={context.urgency_score:.2f}, Anxiety={context.anxiety_level:.2f}, Knowledge={context.knowledge_level}")
            print(f" Adaptation: {context.tone_type} + {context.detail_level}")
            
            # Display response
            print(f"\n MedChat: {response}")
            
            # Update conversation history
            conversation_history.append(user_input)
            conversation_history.append(response)
            
            # Keep only recent history to avoid memory issues
            if len(conversation_history) > 10:
                conversation_history = conversation_history[-10:]
                
        except Exception as e:
            print(f" Error: {e}")
            print("Please try again with a different query.")

# Note: Uncomment the line below to run the interactive demo
interactive_demo()

🩺 MedChat Dynamic Prompting System - Interactive Demo
Enter patient queries to see adaptive responses in real-time.
Type 'quit' to exit the demo.
Type 'examples' to see sample queries.

 Context: Urgency=0.00, Anxiety=0.60, Knowledge=basic
 Adaptation: reassuring_empathetic + low_detail

 MedChat: i can understand why you're feeling this way. it's natural to be scared when you find something unexpected in your body. most lumps are nothing to worry about. they can be caused by things like an infection, inflammation, or even hormonal changes. would you like to video or text chat with me? i'm here to help you understand what this lump could be.

 Context: Urgency=0.00, Anxiety=0.60, Knowledge=basic
 Adaptation: reassuring_empathetic + low_detail

 MedChat: i can understand why you're feeling this way. it's natural to be scared when you find something unexpected in your body. most lumps are nothing to worry about. they can be caused by things like an infection, inflammation, or even hormon